In [ ]:
# === CELL 1: INSTALLATION ONLY ===
# This cell installs everything but does NOT import numpy

import os
import sys

%cd /content
!rm -rf facefusion
!git clone https://github.com/facefusion/facefusion.git
%cd facefusion

# Install all packages
print("Installing packages...")
!pip install -q onnxruntime-gpu==1.23.2
!pip install -q opencv-python==4.12.0.88
!pip install -q onnx==1.19.1
!pip install -q scipy==1.16.3
!pip install -q psutil==7.1.2
!pip install -q tqdm==4.67.1
!pip install -q gradio==5.44.1
!pip install -q gradio-rangeslider==0.0.8

# Force correct numpy as the final step
print("\nForcing numpy==2.2.4...")
!pip uninstall -y numpy 2>&1 | grep -v "WARNING"
!pip install numpy==2.2.4 2>&1 | grep -v "ERROR"

# Verify installation on disk (not in Python)
print("\nVerifying numpy installation on disk...")
!pip show numpy | grep Version

print("\n" + "="*60)
print("✓ INSTALLATION COMPLETE")
print("✓ NOW RUN CELL 2 (DO NOT RUN THIS CELL AGAIN)")
print("="*60)

Adding GRADIO support:

In [ ]:
!pip install gradio==5.44.1 --force-reinstall

%cd /content/facefusion

print("="*60)
print("Patching ONLY layout files (NOT core.py)...")
print("="*60)

# ONLY patch the layout files (these use real Gradio launch)
!sed -i "s/ui.launch(favicon_path/ui.launch(share=True, favicon_path/g" facefusion/uis/layouts/default.py
!sed -i "s/ui.launch(favicon_path/ui.launch(share=True, favicon_path/g" facefusion/uis/layouts/jobs.py
!sed -i "s/ui.launch(favicon_path/ui.launch(share=True, favicon_path/g" facefusion/uis/layouts/webcam.py
#!sed -i "s/ui.launch(favicon_path/ui.launch(share=True, favicon_path/g" facefusion/uis/layouts/benchmark.py
# DO NOT patch core.py - restore it if already patched
#!sed -i "s/ui.launch(share=True)/ui.launch()/g" facefusion/core.py

print("\n" + "="*60)
print("🚀 Launching with PUBLIC GRADIO LINK...")
print("="*60 + "\n")


Adding NSFW OFF

In [ ]:
# === CELL 3: DISABLE NSFW FILTERs ===
%cd /content/facefusion

print("Downloading original files from GitHub...")
!wget -q https://raw.githubusercontent.com/facefusion/facefusion/master/facefusion/content_analyser.py -O facefusion/content_analyser.py
!wget -q https://raw.githubusercontent.com/facefusion/facefusion/master/facefusion/core.py -O facefusion/core.py

print("✓ Fresh files downloaded\n")

# -- HASH check --
# Show original hash check
print("Original hash check in core.py:")
!grep -A 2 "content_analyser_hash = hash_helper" facefusion/core.py

# Remove the hash check from core.py (make it always return True)
!sed -i "s/and content_analyser_hash == 'b14e7b92'//" facefusion/core.py

print("\n✓ Hash check bypassed!")
print("\nModified hash check:")
!grep -A 2 "content_analyser_hash = hash_helper" facefusion/core.py

# -- NSFW models --
print("\n\nDisabling NSFW detection...")
# First, let's see the original function
print("Original function:")
!sed -n '192,199p' facefusion/content_analyser.py

# Use sed to safely replace the line
#!sed -i 's/is_nsfw_1 = detect_with_nsfw_1(vision_frame)/is_nsfw_1 = False/' facefusion/content_analyser.py
!sed -i 's/return is_nsfw_1 and is_nsfw_2 or is_nsfw_1 and is_nsfw_3 or is_nsfw_2 and is_nsfw_3/return False/' facefusion/content_analyser.py

print("\nModified function:")
!sed -n '192,199p' facefusion/content_analyser.py


# Validate Python syntax
print("\nValidating syntax...")
!python -m py_compile facefusion/content_analyser.py && echo "✓ Syntax OK" || echo "✗ Syntax ERROR"

In [ ]:
# Fresh Python environment will load the correct numpy

%cd /content/facefusion

# NOW import numpy (fresh import will get the correct version)
import numpy as np
print(f"Numpy version in Python: {np.__version__}")

# Check if run.py exists
import os
if not os.path.exists('run.py'):
    print("ERROR: run.py not found. Let me check the directory structure...")
    !ls -la
    print("\nChecking if this is the right repo...")
    !cat README.md | head -20
else:
    # Download models if needed
    print("\nSetting up models directory...")
    !mkdir -p .assets/models

    # Launch the application
#      --ui-layouts default \
    print("\nLaunching FaceFusion...")
    !mkdir -p /content/output
    !python run.py \
      --execution-providers cuda \
      --output-path /content/output \
      --processors face_swapper face_enhancer \
      --face-swapper-model inswapper_128 \
      --face-swapper-pixel-boost 256x256 \
      --face-swapper-weight 0.65 \
      --face-enhancer-model codeformer \
      --face-enhancer-blend 66 \
      --face-enhancer-weight 0.65 \
      --face-selector-mode one


In [ ]:
## UPLOAD video manually
# - then just add this new param to run.py
# -t /content/input/target_.mp4 \

!mkdir -p /content/input
!mkdir -p /content/output


# === UPLOAD VIDEO TO COLAB STORAGE ===
from google.colab import files
import shutil

# Upload your video
print("Please select your target video...")
uploaded = files.upload()

# Move to a known location
for filename in uploaded.keys():
    shutil.copy(filename, f'/content/input/target_{os.path.splitext(filename)[1]}')
    print(f"✓ Video saved to: /content/input/target_{os.path.splitext(filename)[1]}")

In [ ]:
'''
import subprocess
import time
import os

# Create output folder
os.makedirs('/content/output', exist_ok=True)

command = [
    "python", "-u", "facefusion.py", "run",
    "--execution-providers", "cuda",
    "--output-path", "/content/output",
    "--ui-layouts", "default",
    "-t", "/content/input/target_.mp4",
    "--processors", "face_swapper", "face_enhancer",
    "--face-swapper-model", "hyperswap_1b_256",
    "--face-swapper-pixel-boost", "512x512",
    "--face-swapper-weight", "0.85",
    "--face-enhancer-model", "codeformer",
    "--face-enhancer-blend", "20",
    "--face-enhancer-weight", "0.04",
    "--face-selector-mode", "one"
]

print("🚀 Starting FaceFusion in background...")

# Open log file with line buffering and don't keep it open
log_file = open("fusion.log", "w", buffering=1)
process = subprocess.Popen(
    command,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp  # Run in separate process group
)

# Wait for URL
print("⏳ Waiting for public URL...")
url_found = False
for i in range(120):
    try:
        with open('fusion.log', 'r') as f:
            content = f.read()
            if 'Running on public URL:' in content:
                for line in content.splitlines():
                    if 'Running on public URL:' in line:
                        print(f"\n✅ {line}\n")
                        url_found = True
                        break
                if url_found:
                    break
    except:
        pass
    time.sleep(2)

if url_found:
    print("✅ FaceFusion is running! Now run the next cell for auto-download.")
else:
    print("⚠️ URL not found yet, but FaceFusion should be starting...")

print(f"💡 Process ID: {process.pid}")
'''

In [ ]:
'''
import os
import time
from google.colab import files
from IPython.display import display, Javascript

output_folder = '/content/output'
tracking_file = '/content/downloaded_files.txt'

if os.path.exists(tracking_file):
    with open(tracking_file, 'r') as f:
        downloaded_files = set(f.read().splitlines())
else:
    downloaded_files = set()

print(f"👀 Watching {output_folder}... ({len(downloaded_files)} already downloaded)")

found_new = False

while not found_new:
    if os.path.exists(output_folder):
        current_files = [f for f in os.listdir(output_folder) if f.endswith('.mp4')]

        for filename in current_files:
            filepath = os.path.join(output_folder, filename)

            if filename not in downloaded_files and os.path.isfile(filepath):
                initial_size = os.path.getsize(filepath)
                time.sleep(2)
                if os.path.getsize(filepath) == initial_size and initial_size > 0:
                    print(f"\n🎬 New: {filename} ({initial_size / 1024 / 1024:.1f} MB)")

                    downloaded_files.add(filename)
                    with open(tracking_file, 'w') as f:
                        f.write('\n'.join(downloaded_files))

                    files.download(filepath)
                    found_new = True
                    break

    if not found_new:
        time.sleep(6)

print("✅ Download triggered! Waiting 3 seconds, then restarting watcher...")
time.sleep(3)

# Auto-restart this cell
display(Javascript('Jupyter.notebook.execute_cell_range(Jupyter.notebook.get_selected_index(), Jupyter.notebook.get_selected_index()+1)'))
'''

In [ ]:
'''
## UPLOAD video manually
# - then just add this new param to run.py
# -t /content/input/target_.mp4 \

!mkdir -p /content/input
!mkdir -p /content/output


# === UPLOAD VIDEO TO COLAB STORAGE ===
from google.colab import files
import shutil

# Upload your video
print("Please select your target video...")
uploaded = files.upload()

# Move to a known location
for filename in uploaded.keys():
    shutil.copy(filename, f'/content/input/target_{os.path.splitext(filename)[1]}')
    print(f"✓ Video saved to: /content/input/target_{os.path.splitext(filename)[1]}")
'''

In [ ]:
#'''
# Launch
!mkdir -p /content/input
!mkdir -p /content/output
%cd /content/facefusion

!python facefusion.py run \
  --execution-providers cuda \
  --output-path /content/output \
  -t /content/input/target_.mp4 \
  --processors face_swapper face_enhancer \
  --face-swapper-model hyperswap_1c_256 \
  --face-swapper-pixel-boost 512x512 \
  --face-swapper-weight 0.65 \
  --face-enhancer-model gpen_bfr_512 \
  --face-enhancer-blend 12 \
  --face-enhancer-weight 0.1 \
  --face-selector-mode one \
  --face-detector-score 0.65 \
  --face-landmarker-score 0.65 \
  --face-detector-angles 0 90 270 \
  --face-mask-blur 0.35 \
  --face-mask-types box occlusion

#'''

In [ ]:
'''
import os
import signal

pid = 2650

try:
    os.kill(pid, signal.SIGKILL)  # Force kill
    print(f"Process {pid} killed")
except ProcessLookupError:
    print(f"Process {pid} not found")
'''

In [ ]:
'''
import subprocess
import time
import os

try:
        with open('fusion.log', 'r') as f:
            content = f.read()
            if 'Running on public URL:' in content:
                for line in content.splitlines():
                    if 'Running on public URL:' in line:
                        print(f"\n✅ {line}\n")
                        url_found = True
            else:
                print("⚠️ URL not found yet, but FaceFusion should be starting...")
except:
        pass
'''

In [ ]:
import os
from pathlib import Path

def find_file(search_dir, target_filename):
    """
    Search for a file in all subfolders of the given directory.

    Args:
        search_dir (str): The root directory to search in
        target_filename (str): The filename to search for

    Returns:
        list: List of full paths to matching files, or empty list if not found
    """
    matching_files = []

    try:
        # Use rglob to recursively search all subdirectories
        for file_path in Path(search_dir).rglob(target_filename):
            matching_files.append(str(file_path))
    except PermissionError:
        print(f"Permission denied accessing {search_dir}")
    except Exception as e:
        print(f"Error during search: {e}")

    return matching_files

# Usage
if __name__ == "__main__":
    search_directory = "/tmp/gradio"
    target_file = "94cc28d0.mp4"

    results = find_file(search_directory, target_file)

    if results:
        print(f"Found {len(results)} file(s):")
        for file_path in results:
            print(f"  {file_path}")
    else:
        print(f"File '{target_file}' not found in {search_directory}")